# 14 — Diagnose the broken harness
Eight small faults represent capability, context, memory, intelligence,
orchestration, control, and verification. Pair up: fix one or two faults,
run the checks again, then compare diagnoses with the room.



**Finale after lesson 13: fix the broken harness.**


## Problem

Agent failures can arise in different harness layers but look similar at the output.


## Concept

Use a failing check to identify the boundary, then make the smallest repair.

A visible failure may originate in a different layer from the final answer. A reproducible test identifies the violated contract before a targeted repair is made.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out   ◀ YOU ARE HERE
   │
   ├─ 06 MEMORY          recall this customer's saved constraint   ◀ YOU ARE HERE
   ├─ 09 CONTROL         validate the question before any model call   ◀ YOU ARE HERE
   ├─ 08 FLOW            typed state · route · one bounded retry   ◀ YOU ARE HERE
   │    └─ 07 CREW            Researcher → Analyst → Critic   ◀ YOU ARE HERE
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer   ◀ YOU ARE HERE
   │              ├─ 01 MODEL          the LLM call itself   ◀ YOU ARE HERE
   │              ├─ 03 TOOLS          typed Python functions the agent may call   ◀ YOU ARE HERE
   │              ├─ 04 MCP            tools served by a separate process   ◀ YOU ARE HERE
   │              └─ 05 KNOWLEDGE      cited passages from company documents   ◀ YOU ARE HERE
   └─ 09 CONTROL         validate the report → human approval → mock email   ◀ YOU ARE HERE

 around everything:
   10 RESILIENCE      checkpoint and resume   ◀ YOU ARE HERE
   11 OBSERVABILITY   trace what actually ran   ◀ YOU ARE HERE
   12 EVALUATION      checks that fail loudly   ◀ YOU ARE HERE
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
symptom ──▶ failing test ──▶ trace responsible boundary
                                  │
                      minimal repair + rerun
                                  │
                      no new regression? ──▶ explain
```

symptom → failing case → responsible layer → repair → rerun. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Load the seeded-fault examples and run the indicated checks

Load the seeded-fault examples and run the indicated checks.


In [ ]:
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.control import validate_report
from workshop.schemas import DecisionReport, Evidence
from workshop.tools import calculate_annual_roi


def choose_tool(question: str) -> str:
    """Fault 1: ROI questions should use the calculator."""
    return "Search company documents"  # BUG


def bad_roi(benefit: float, cost: float) -> float:
    """Fault 2: invalid cost should not produce an answer."""
    return 0.0 if cost == 0 else calculate_annual_roi(benefit, cost)  # BUG


def cited_report() -> DecisionReport:
    """Fault 3: one source ID is not in the sample corpus."""
    return DecisionReport(
        recommendation="pilot",
        rationale="A pilot may be justified if missing security evidence is gathered first.",
        evidence=[
            Evidence(source_id="architecture.md", claim="Private deployment is possible."),
            Evidence(source_id="vendor-blog.md", claim="Deletion is guaranteed."),  # BUG
        ],
    )


def memory_scope(customer_id: str) -> str:
    """Fault 4: all customers are mapped to the same scope."""
    return "/customer/shared"  # BUG


def conclusion(report: DecisionReport) -> str:
    """Fault 5: full adoption is asserted despite unresolved evidence."""
    return "adopt"  # BUG


def research_attempts(max_retry: int) -> int:
    """Fault 6: the configured retry cap is ignored."""
    return max_retry + 5  # BUG


def may_send_email(approved: bool) -> bool:
    """Fault 7: a side effect can happen without approval."""
    return True  # BUG


def trace_events() -> list[str]:
    """Fault 8: tool use is missing from the trace."""
    return ["CrewKickoffStartedEvent", "CrewKickoffCompletedEvent"]  # BUG


def rejects_invalid_cost() -> bool:
    try:
        bad_roi(45_000, 0)
    except ValueError:
        return True
    return False




## Run the failure checks
A failing check is the clue, not the solution. Use the function, its boundary,
and what the earlier labs taught to make the smallest justified repair.



## Step 2 — Change only the responsible layer, rerun the checks, and explain the regression prevented

Change only the responsible layer, rerun the checks, and explain the regression prevented.


In [ ]:
checks = {
    "1 tool choice": choose_tool("Calculate annual ROI") == "Calculate annual ROI",
    "2 input contract": rejects_invalid_cost(),
    "3 source validation": validate_report(cited_report())[0],
    "4 memory isolation": memory_scope("A") != memory_scope("B"),
    "5 supported conclusion": conclusion(cited_report()) != "adopt",
    "6 bounded retries": research_attempts(1) <= 2,
    "7 approval boundary": not may_send_email(False),
    "8 tool trace": "ToolUsageFinishedEvent" in trace_events(),
}
for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), name)


## Failure mode to know about

Fixing the visible answer without repairing the responsible layer can hide the fault. Reproduce, isolate the boundary, patch, and rerun the relevant regression checks.


## Production implication

Turn each discovered fault into a regression test with a named boundary. Recheck adjacent boundaries so the repair does not move the failure elsewhere.


## Your turn

Fix one or two assigned faults and explain the evidence behind your diagnosis.

**Checkpoint:** Your assigned checks pass and the repair does not weaken another boundary.
